# Front optics and pump generation — release 0.2

This notebook is the first optical layer: coherent normal-incidence propagation
through thin POx/AlOx films followed by Beer–Lambert absorption in a thick InP
substrate. It answers how much pump power enters the wafer and where carriers
are generated. It deliberately does **not** yet claim a PL collection efficiency
or photon-recycling factor.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / "src").exists(): root = root.parent
sys.path.insert(0, str(root / "src"))
from surpass import (BulkModel, Layer, OpticalStack, Wafer,
                        get_optical_material, single_pass_escape_probability)

try:
    import scienceplots
    plt.style.use(["science", "notebook", "no-latex"])
except ImportError:
    plt.style.use("default")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## 1. Editable optical experiment

The film thicknesses and optical constants below are placeholders. Replace the
demo dispersions with measured ellipsometry before quantitative fitting.

In [ ]:
pump_wavelength_nm = 514.0
wafer_thickness_um = 620.0
pox_thickness_nm = 10.0
alox_thickness_nm = 10.0
intensity_n_w_cm2 = 1.0
intensity_p_w_cm2 = 13.0

air = get_optical_material("air")
inp = get_optical_material("InP_demo")
pox = get_optical_material("POx_demo")
alox = get_optical_material("AlOx_demo")

bare = OpticalStack(air, [], inp)
coated = OpticalStack(air, [Layer(pox, pox_thickness_nm, "POx"),
                            Layer(alox, alox_thickness_nm, "AlOx")], inp)

for item in (inp, pox, alox):
    print(f"{item.name}: N({pump_wavelength_nm:g} nm)={item.nk(pump_wavelength_nm)}, "
          f"status={item.status}")
    print("  source:", item.source)
    if item.note: print("  caution:", item.note)

## 2. Energy-balance and absorption-depth checks

In [ ]:
rows = []
for name, stack in [("bare InP", bare), ("POx/AlOx/InP", coated)]:
    result = stack.solve(pump_wavelength_nm)
    rows.append({"stack": name, "R": result.reflectance,
                 "power entering InP": result.substrate_entry_fraction,
                 "thin-film absorption": sum(result.layer_absorptance),
                 "balance error": result.balance_error,
                 "InP alpha (cm^-1)": result.substrate_alpha_cm1,
                 "1/e depth (nm)": result.absorption_depth_nm})
display(pd.DataFrame(rows).style.format(precision=5))

## 3. Pump-generation profiles

In [ ]:
depth_nm = np.linspace(0, 1500, 1501)
depth_cm = depth_nm*1e-7
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for stack_name, stack, intensity, color in [
    ("bare, 1 W cm$^{-2}$", bare, intensity_n_w_cm2, "black"),
    ("coated, 1 W cm$^{-2}$", coated, intensity_n_w_cm2, "tab:blue"),
    ("coated, 13 W cm$^{-2}$", coated, intensity_p_w_cm2, "tab:red")]:
    generation = stack.generation_profile_cm3_s(pump_wavelength_nm, intensity, depth_cm)
    axes[0].plot(depth_nm, generation, label=stack_name, color=color)
    axes[1].semilogy(depth_nm, generation, label=stack_name, color=color)
for ax in axes:
    ax.set(xlabel="Depth into InP (nm)", ylabel="$G(z)$ (cm$^{-3}$ s$^{-1}$)")
axes[0].legend(fontsize=8)
axes[0].set_title("Linear scale"); axes[1].set_title("Log scale")
fig.tight_layout()

## 4. Optical depth versus transport length scales

In [ ]:
comparison = []
for label, wafer in {
    "n-InP": Wafer("InP", donor_cm3=8.5e18, thickness_um=wafer_thickness_um),
    "p-InP": Wafer("InP", acceptor_cm3=5.1e18, thickness_um=wafer_thickness_um),
}.items():
    state = BulkModel(wafer, tau_srh_s=100e-9).injection(1e15)
    comparison.append({"wafer": label, "diffusion length at dn=1e15 (um)": state.diffusion_length_um,
                       "pump 1/e depth (um)": coated.solve(pump_wavelength_nm).absorption_depth_nm/1000,
                       "wafer thickness (um)": wafer_thickness_um})
display(pd.DataFrame(comparison).style.format(precision=3))

The pump is absorbed near the front surface, while a diffusion length can be
orders of magnitude larger. Consequently, a rear boundary condition cannot be
chosen from the optical absorption depth alone; the later transport solver must
compare wafer thickness, injection-dependent diffusion length, and both surface
recombination velocities.

## 5. A bounded escape-cone check (not yet a PL model)

In [ ]:
emission_wavelength_nm = 920.0
n_emission = inp.nk(emission_wavelength_nm).real
p_escape = single_pass_escape_probability(n_emission)
display(pd.DataFrame({
    "emission wavelength (nm)": [emission_wavelength_nm],
    "demo InP index": [n_emission],
    "single-pass escape probability": [p_escape],
}))

This escape value integrates Fresnel transmission over one planar escape cone
and normalizes to the full isotropic photon population. It excludes reabsorption,
re-emission, repeated reflections, thin-film interference at emission
wavelengths, free-carrier absorption, and collection numerical aperture.

### Next optical/transport increment

1. replace demo $n,k$ with tabulated or measured spectra;
2. add finite-wafer incoherent rear reflections;
3. solve 1D generation–diffusion–recombination with surface boundary conditions;
4. add wavelength-resolved spontaneous emission, reabsorption, and photon
   recycling; and
5. integrate the emitted flux through the actual collection optics for PL and
   linearize the time-dependent solver for TRPL.